# Ensemble Tutorial: Comparing LRC and Communities Across Datasets

This notebook extends the earlier single-dataset comparison into a cross-dataset tutorial.

We compare five supported ensemble classifiers with DPG:

- `BaggingClassifier`
- `ExtraTreesClassifier`
- `AdaBoostClassifier`
- `RandomForestClassifier`
- `GradientBoostingClassifier`

and we run the same workflow on four datasets:

- `iris`
- `wine`
- `breast_cancer`
- `wheat_seeds`

The goal is not just to ask which model is most accurate, but to compare how the **DPG structure** changes across datasets:

- which predicates rise to the top under **Local Reaching Centrality (LRC)**
- how many **communities** appear
- whether community structure is balanced or compressed
- how graph size changes from easy multiclass problems to denser, higher-dimensional ones

The notebook stays tutorial-oriented: it favors readable helper functions, compact tables, and discussion cells over a large benchmarking framework.


In [ ]:
from __future__ import annotations

import io
import os
import sys
import urllib.request
from contextlib import redirect_stderr, redirect_stdout
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR
if not (PROJECT_ROOT / "dpg").exists():
    for candidate in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
        if (candidate / "dpg").exists():
            PROJECT_ROOT = candidate
            break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer, load_iris, load_wine
from sklearn.ensemble import (
    AdaBoostClassifier,
    BaggingClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

from dpg import DPGExplainer, class_feature_predicate_counts
from dpg.visualizer import lrc_predicate_scores

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 200)
plt.rcParams["figure.dpi"] = 120

SEED = 27
CACHE_DIR = PROJECT_ROOT / "tutorials" / ".cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)


## 1. Load the Datasets

The four datasets were chosen because they stress different kinds of decision structure.

- `iris`: small, clean, low-dimensional multiclass baseline
- `wine`: multiclass with richer continuous chemistry features
- `breast_cancer`: higher-dimensional binary classification
- `wheat_seeds`: compact 3-class geometric dataset with meaningful overlap between varieties

`wheat_seeds` is fetched from the UCI Seeds dataset and cached locally inside `tutorials/.cache/` the first time the notebook runs.


In [ ]:
def load_wheat_seeds(cache_dir: Path = CACHE_DIR) -> tuple[pd.DataFrame, pd.Series, list[str]]:
    cache_path = cache_dir / "seeds_dataset.txt"
    if not cache_path.exists():
        url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00236/seeds_dataset.txt"
        urllib.request.urlretrieve(url, cache_path)

    col_names = [
        "area",
        "perimeter",
        "compactness",
        "kernel_length",
        "kernel_width",
        "asymmetry_coeff",
        "groove_length",
        "variety",
    ]
    raw = pd.read_csv(cache_path, sep=r"\s+", header=None, names=col_names)
    X = raw[col_names[:-1]].copy()
    y = raw["variety"].astype(int) - 1
    class_names = ["Kama", "Rosa", "Canadian"]
    return X, y, class_names


def load_datasets() -> dict[str, dict[str, object]]:
    datasets = {}

    iris = load_iris(as_frame=True)
    datasets["iris"] = {
        "X": iris.data.round(3),
        "y": iris.target,
        "class_names": iris.target_names.tolist(),
        "notes": "3 classes, 4 features, very compact structure",
    }

    wine = load_wine(as_frame=True)
    datasets["wine"] = {
        "X": wine.data.round(3),
        "y": wine.target,
        "class_names": wine.target_names.tolist(),
        "notes": "3 classes, 13 features, richer threshold structure",
    }

    cancer = load_breast_cancer(as_frame=True)
    datasets["breast_cancer"] = {
        "X": cancer.data.round(3),
        "y": cancer.target,
        "class_names": cancer.target_names.tolist(),
        "notes": "2 classes, 30 features, larger feature space",
    }

    wheat_X, wheat_y, wheat_class_names = load_wheat_seeds()
    datasets["wheat_seeds"] = {
        "X": wheat_X.round(3),
        "y": wheat_y,
        "class_names": wheat_class_names,
        "notes": "3 classes, 7 features, geometric grain morphology",
    }

    return datasets


datasets = load_datasets()

dataset_inventory = pd.DataFrame(
    [
        {
            "dataset": name,
            "n_samples": payload["X"].shape[0],
            "n_features": payload["X"].shape[1],
            "n_classes": len(payload["class_names"]),
            "notes": payload["notes"],
        }
        for name, payload in datasets.items()
    ]
).sort_values("dataset").reset_index(drop=True)

dataset_inventory


## 2. Define the Ensemble Models

We keep one compact configuration per ensemble family so the notebook remains easy to run.

The purpose is comparative interpretation rather than hyperparameter tuning.


In [ ]:
def make_bagging() -> BaggingClassifier:
    tree = DecisionTreeClassifier(max_depth=4, random_state=SEED)
    try:
        return BaggingClassifier(estimator=tree, n_estimators=8, random_state=SEED)
    except TypeError:
        return BaggingClassifier(base_estimator=tree, n_estimators=8, random_state=SEED)


def make_adaboost() -> AdaBoostClassifier:
    tree = DecisionTreeClassifier(max_depth=2, random_state=SEED)
    try:
        return AdaBoostClassifier(estimator=tree, n_estimators=8, random_state=SEED)
    except TypeError:
        return AdaBoostClassifier(base_estimator=tree, n_estimators=8, random_state=SEED)


model_builders = {
    "Bagging": make_bagging,
    "Extra Trees": lambda: ExtraTreesClassifier(
        n_estimators=8,
        max_depth=4,
        random_state=SEED,
        n_jobs=-1,
    ),
    "AdaBoost": make_adaboost,
    "Random Forest": lambda: RandomForestClassifier(
        n_estimators=8,
        max_depth=4,
        random_state=SEED,
        n_jobs=-1,
    ),
    "Gradient Boosting": lambda: GradientBoostingClassifier(
        n_estimators=8,
        max_depth=3,
        random_state=SEED,
    ),
}

list(model_builders)


## 3. Helper Functions

The next helper block performs the same workflow for each `(dataset, model)` pair:

- train/test split
- fit the ensemble
- build the DPG explanation
- extract top-LRC predicates
- summarize community sizes and feature themes

This keeps the rest of the notebook focused on reading the outputs.


In [ ]:
def count_predicates(explanation) -> int:
    labels = explanation.node_metrics["Label"].astype(str)
    predicate_mask = labels.str.contains("<=", regex=False) | labels.str.contains(">", regex=False)
    return int(predicate_mask.sum())


def class_node_count(explanation) -> int:
    labels = explanation.node_metrics["Label"].astype(str)
    return int(labels.str.startswith("Class ").sum())


def community_size_table(explanation, dataset_name: str, model_name: str) -> pd.DataFrame:
    clusters = {}
    if explanation.communities is not None:
        clusters = explanation.communities.get("Clusters", {})
    rows = []
    for label, members in clusters.items():
        rows.append(
            {
                "dataset": dataset_name,
                "model": model_name,
                "community_label": label,
                "n_nodes": len(members),
            }
        )
    return pd.DataFrame(rows)


def community_feature_table(explanation, dataset_name: str, model_name: str, top_n: int = 3) -> pd.DataFrame:
    heat_df = class_feature_predicate_counts(explanation)
    rows = []
    for community_label, counts in heat_df.iterrows():
        top_features = counts.sort_values(ascending=False)
        top_features = top_features[top_features > 0].head(top_n)
        rows.append(
            {
                "dataset": dataset_name,
                "model": model_name,
                "community_label": str(community_label),
                "top_features": ", ".join(
                    f"{feature} ({int(value)})" for feature, value in top_features.items()
                ),
            }
        )
    return pd.DataFrame(rows)


def fit_and_explain_dataset_model(dataset_name: str, payload: dict, model_name: str, builder) -> dict:
    X = payload["X"]
    y = payload["y"]
    class_names = payload["class_names"]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.30,
        random_state=SEED,
        stratify=y,
    )

    model = builder()
    model.fit(X_train, y_train)
    accuracy = accuracy_score(y_test, model.predict(X_test))

    explainer = DPGExplainer(
        model=model,
        feature_names=X_train.columns.tolist(),
        target_names=list(class_names),
    )

    sink = io.StringIO()
    with redirect_stdout(sink), redirect_stderr(sink):
        explanation = explainer.explain_global(X_train.values, communities=True)

    lrc_df = lrc_predicate_scores(explanation, top_k=10)
    community_sizes = community_size_table(explanation, dataset_name, model_name)
    community_features = community_feature_table(explanation, dataset_name, model_name)

    summary = {
        "dataset": dataset_name,
        "model": model_name,
        "test_accuracy": round(float(accuracy), 3),
        "n_samples_train": int(X_train.shape[0]),
        "n_features": int(X_train.shape[1]),
        "n_classes": int(len(class_names)),
        "n_nodes": int(len(explanation.nodes)),
        "n_predicates": count_predicates(explanation),
        "n_class_nodes": class_node_count(explanation),
        "n_communities": int(len(community_sizes)),
        "top_predicate": lrc_df.iloc[0]["predicate"] if not lrc_df.empty else None,
        "top_feature": lrc_df.iloc[0]["feature"] if not lrc_df.empty else None,
        "top_lrc": round(float(lrc_df.iloc[0]["lrc"]), 3) if not lrc_df.empty else np.nan,
        "mean_top5_lrc": round(float(lrc_df.head(5)["lrc"].mean()), 3) if not lrc_df.empty else np.nan,
    }

    return {
        "model": model,
        "explainer": explainer,
        "explanation": explanation,
        "summary": summary,
        "lrc_df": lrc_df,
        "community_sizes": community_sizes,
        "community_features": community_features,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "class_names": class_names,
    }


## 4. Run the Cross-Dataset Comparison

This will build one DPG explanation for every combination of:

- 4 datasets
n- 5 ensemble models

for a total of 20 tutorial-sized runs.


In [ ]:
results = {}
for dataset_name, payload in datasets.items():
    for model_name, builder in model_builders.items():
        results[(dataset_name, model_name)] = fit_and_explain_dataset_model(
            dataset_name,
            payload,
            model_name,
            builder,
        )

summary_df = pd.DataFrame([payload["summary"] for payload in results.values()])
summary_df.sort_values(["dataset", "test_accuracy", "n_nodes"], ascending=[True, False, True]).reset_index(drop=True)


## 5. Compare Accuracy and Graph Size Across Datasets

The first useful comparison is to read performance together with graph size.

Interpretation guideline:

- high accuracy with a small graph can indicate a cleanly separable dataset
- high accuracy with a large graph can indicate a more distributed decision logic
- lower accuracy with a large graph may suggest the model is working harder on a harder boundary


In [ ]:
accuracy_pivot = summary_df.pivot(index="dataset", columns="model", values="test_accuracy")
node_pivot = summary_df.pivot(index="dataset", columns="model", values="n_nodes")
community_pivot = summary_df.pivot(index="dataset", columns="model", values="n_communities")

print("Accuracy")
display(accuracy_pivot.round(3))
print("\nDPG node count")
display(node_pivot.astype(int))
print("\nCommunity count")
display(community_pivot.astype(int))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

accuracy_pivot.plot(kind="bar", ax=axes[0], title="Test accuracy by dataset and model")
axes[0].set_ylim(0.0, 1.05)
axes[0].set_ylabel("accuracy")
axes[0].tick_params(axis="x", rotation=45)

node_pivot.plot(kind="bar", ax=axes[1], title="DPG node count by dataset and model")
axes[1].set_ylabel("number of nodes")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


### Cross-Dataset Reading

A robust reading should distinguish **dataset difficulty** from **model family behavior**.

Expected patterns in this set:

- `iris` usually produces smaller graphs and cleaner communities because the classes are well separated.
- `wine` often produces richer threshold structure because it has more continuous features and more chemically meaningful cut points.
- `breast_cancer` can produce large graphs despite being binary, simply because it has many features and several correlated measurement families.
- `wheat_seeds` is often a good middle ground: still compact, but with more overlap than `iris`, so communities and top-LRC predicates become more informative.

Also compare model families carefully:

- `Random Forest` and `Extra Trees` often preserve more structural diversity, which can increase graph size.
- `AdaBoost` often gives a more compact graph.
- `Bagging` often sits between a compact boosted structure and a richer forest structure.
- `Gradient Boosting` can still be highly accurate, but its DPG logic may look more stage-wise or concentrated depending on the dataset.


## 6. Compare the Top LRC Predicates Across All Runs

LRC is a structural importance measure over predicates, not just over raw features.

This means two models may agree that a feature matters while disagreeing on:

- the threshold value
- the branch direction
- how early and globally that predicate shapes the decision graph


In [ ]:
top_lrc_table = pd.concat(
    [
        payload["lrc_df"].head(5).assign(dataset=dataset_name, model=model_name)
        for (dataset_name, model_name), payload in results.items()
    ],
    ignore_index=True,
)

top_lrc_table[["dataset", "model", "predicate", "feature", "lrc"]].sort_values(
    ["dataset", "model", "lrc"],
    ascending=[True, True, False],
).reset_index(drop=True)


In [ ]:
top_feature_counts = (
    summary_df.groupby(["dataset", "top_feature"]) 
    .size()
    .rename("models_where_feature_is_top_LRC")
    .reset_index()
    .sort_values(["dataset", "models_where_feature_is_top_LRC", "top_feature"], ascending=[True, False, True])
)

top_feature_counts


In [ ]:
for dataset_name in sorted(datasets):
    subset = summary_df[summary_df["dataset"] == dataset_name].sort_values("top_lrc", ascending=False)
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.barh(subset["model"], subset["top_lrc"], color="#4C78A8")
    ax.set_title(f"{dataset_name}: top LRC score by model")
    ax.set_xlabel("top predicate LRC")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()


### LRC Discussion

This is where the dataset differences become more concrete.

- On `iris`, top-LRC predicates often concentrate on a small set of petal-based splits. Agreement across models is usually high.
- On `wine`, agreement is often partial rather than complete: several models may highlight `flavanoids`, `proline`, `color_intensity`, or `od280/od315_of_diluted_wines`, but not always with the same threshold.
- On `breast_cancer`, top predicates often come from a small subset of highly discriminative measurement families, even though the full graph can still be large.
- On `wheat_seeds`, geometric variables such as `area`, `kernel_length`, `kernel_width`, or `groove_length` often become the structural anchors.

A useful robustness question is:

> do different ensemble families keep returning the same top feature on the same dataset?

If yes, that is evidence of stable structural signal. If not, the graph may be revealing multiple equally plausible ways to separate the classes.


## 7. Compare Communities Across All Datasets

Community summaries are especially useful when we stop reading the graph as a list of single predicates and start reading it as **decision themes**.

The next cells aggregate the community outputs across all `(dataset, model)` runs.


In [ ]:
community_sizes_df = pd.concat(
    [payload["community_sizes"] for payload in results.values()],
    ignore_index=True,
)
community_features_df = pd.concat(
    [payload["community_features"] for payload in results.values()],
    ignore_index=True,
)

print("Community sizes")
display(community_sizes_df.sort_values(["dataset", "model", "n_nodes"], ascending=[True, True, False]).reset_index(drop=True))

print("\nTop community features")
display(community_features_df.sort_values(["dataset", "model", "community_label"]).reset_index(drop=True))


In [ ]:
community_count_view = summary_df.pivot(index="dataset", columns="model", values="n_communities")
community_count_view.plot(
    kind="bar",
    figsize=(10, 4),
    title="Community count by dataset and model",
    color=["#4C78A8", "#F58518", "#54A24B", "#E45756", "#72B7B2"],
)
plt.ylabel("number of communities")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
for dataset_name in sorted(datasets):
    subset = community_sizes_df[community_sizes_df["dataset"] == dataset_name]
    if subset.empty:
        continue
    pivot = subset.pivot_table(index="model", columns="community_label", values="n_nodes", fill_value=0)
    pivot.plot(kind="bar", stacked=True, figsize=(10, 4), title=f"{dataset_name}: community sizes by model")
    plt.ylabel("number of nodes")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


### Community Discussion

There are two different questions to ask here.

1. **How many communities appear?**

A larger count can mean richer class-specific structure, but it can also mean fragmentation.
A smaller count can mean elegant compression, but it can also mean that many decisions were absorbed into a broad shared or ambiguous region.

2. **What features dominate each community?**

This is often more informative than the raw count. If a community is consistently organized around a few strong features, the model is expressing a recognizable decision theme.

Dataset-specific reading:

- `iris`: communities are often clean and class-aligned, because one or two features dominate separation.
- `wine`: communities are usually more feature-rich, and different models may allocate chemistry features to communities differently.
- `breast_cancer`: because the task is binary, community counts are naturally lower, so feature composition matters more than raw count.
- `wheat_seeds`: communities can be especially interpretable because the features have direct geometric meaning.

Model-specific reading:

- Forest-style ensembles often preserve more alternative paths, which can enlarge community structure.
- Boosting can create more concentrated rule ecosystems.
- If `Ambiguous` appears, it is not necessarily a failure: it often means some predicates are structurally shared rather than class-exclusive.


## 8. Dataset-by-Dataset Summary Tables

This section makes it easier to read each dataset separately before jumping to a final cross-dataset takeaway.


In [ ]:
for dataset_name in sorted(datasets):
    print(dataset_name.upper())
    display(
        summary_df[summary_df["dataset"] == dataset_name]
        .sort_values(["test_accuracy", "n_nodes"], ascending=[False, True])
        .reset_index(drop=True)
    )


## 9. Optional Deep Dive

Pick one dataset and one model to inspect the raw LRC table and the community feature matrix.


In [ ]:
selected_dataset = "wine"
selected_model = "Gradient Boosting"
selected = results[(selected_dataset, selected_model)]

print("Summary")
display(pd.DataFrame([selected["summary"]]))

print("\nTop LRC predicates")
display(selected["lrc_df"].head(10))

print("\nCommunity sizes")
display(selected["community_sizes"])

print("\nCommunity feature counts")
display(class_feature_predicate_counts(selected["explanation"]))


## 10. Final Takeaways

A robust cross-dataset interpretation usually looks like this:

- First read **accuracy** to verify that the model is solving the task reasonably well.
- Then read **graph size** to estimate how much structural complexity the model is carrying.
- Then inspect **LRC** to identify the predicates that globally organize the graph.
- Finally inspect **communities** to understand whether those predicates form clean class-specific ecosystems or a more shared structure.

Across these four datasets, the key lesson is that DPG is not only comparing models, it is also comparing **how much structure the dataset itself demands**.

Practical conclusions:

- Use `iris` as the clean baseline for understanding the workflow.
- Use `wine` when you want a richer multiclass comparison where model families can meaningfully disagree.
- Use `breast_cancer` to study what happens when feature dimensionality increases even in a binary task.
- Use `wheat_seeds` when you want a compact multiclass dataset whose communities remain human-readable.

For model families:

- `Random Forest` and `Extra Trees` are often strong references when you want richer structural explanations.
- `AdaBoost` can be a good compact baseline.
- `Bagging` often offers a useful middle point.
- `Gradient Boosting` is now included in the same workflow and is especially interesting to compare when you want to see how stage-wise boosting logic differs from bagging-style ensembles.
